# RNN & LSTM — The Library Version

Two honest demonstrations: (1) the fixed-width contrast — a strong sklearn MLP on flattened parity, which can FIT a fixed length but structurally cannot ACCEPT any other; (2) the PyTorch translation, shown not run — every line maps to a block you wrote.

In [1]:
import numpy as np
import pandas as pd
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split

df = pd.read_csv("data/parity_data.csv", dtype={"bits": str})
B = np.array([[int(ch) for ch in s] for s in df.bits])
y = df.parity.values
Xf = B.astype(float)                                   # flattened: 12 columns, order discarded

Xtr, Xte, ytr, yte = train_test_split(Xf, y, test_size=0.25, random_state=0)
mlp = MLPClassifier(hidden_layer_sizes=(64, 64), max_iter=3000, random_state=0).fit(Xtr, ytr)
print(f"MLP on flattened T=12 parity — train: {mlp.score(Xtr, ytr):.0%}   test: {mlp.score(Xte, yte):.0%}")
print("(parity is brutally hard for feedforward nets: it must be memorized region by region;")
print(" whatever the score, note it took 2x64 hidden units vs the scratch RNN's 12)")
print()
try:
    B30 = np.random.default_rng(1).integers(0, 2, (5, 30)).astype(float)
    mlp.predict(B30)
except Exception as e:
    print(f"MLP fed a length-30 string -> {type(e).__name__}: {str(e)[:60]}...")
    print()
    print("Not a low score — a REFUSAL. Fixed-width models cannot even accept the input.")
    print("The scratch RNN, trained on T=12, scored ~97% at T=80 (from-scratch notebook, Block 5).")

MLP on flattened T=12 parity — train: 100%   test: 72%
(parity is brutally hard for feedforward nets: it must be memorized region by region;
 whatever the score, note it took 2x64 hidden units vs the scratch RNN's 12)

MLP fed a length-30 string -> ValueError: X has 30 features, but MLPClassifier is expecting 12 feature...

Not a low score — a REFUSAL. Fixed-width models cannot even accept the input.
The scratch RNN, trained on T=12, scored ~97% at T=80 (from-scratch notebook, Block 5).


### The PyTorch translation — read it; you have built every line

```python
import torch.nn as nn

rnn  = nn.RNN(input_size=1,  hidden_size=24, batch_first=True)   # Block 3's loop (tanh cell)
lstm = nn.LSTM(input_size=2, hidden_size=24, batch_first=True)   # Block 7, all four gates
head = nn.Linear(24, 1)                                           # the Wy/by readout

out, h_n        = rnn(x)     # out: every step's h  (our hs[:, 1:])
out, (h_n, c_n) = lstm(x)    # h AND the belt c — the tuple is Block 7's (h, c)
```

`loss.backward()` runs Blocks 4 and 7's backward passes — BPTT and the four-gate chain rule — generated automatically. PyTorch's LSTM even ships the same trick our `lstm_init` used: forget-gate bias initialization is a documented flag, because "remember by default" matters that much in practice.

**The takeaway:** the gap between this folder and production sequence models of the RNN era is autograd + GPUs + stacking (multi-layer, bidirectional). The cell you built IS the cell. And the two limits you measured in Block 10 — the sequential tax and the fixed-size baton — are precisely why the field moved to lesson 13's attention.